# Exercise 2 — Code Generation with ReACT Prompting

**Tools used:** ChatGPT for ReACT-style code generation, Google Colab for execution/testing, GitHub for notebook sharing.

**Goal:** Generate and improve Python code through a Plan → Act → Run → Observe → Fix cycle.

### Full ReACT-style prompt
> **Plan:** First give a short implementation plan, not hidden chain-of-thought. The task is to build a customer-segmentation script in Python from a small DataFrame containing annual spending, purchase frequency, age, and region.  
> **Act:** Generate runnable Python using only pandas, NumPy, scikit-learn, and matplotlib if needed. Validate required numeric columns, handle missing numeric values, standardize the features, evaluate K values from 2 to 4 with silhouette score, select the best K, fit K-Means with `random_state=42` and explicit `n_init`, print the evaluation scores and a cluster summary, and save the final rows to a CSV file.  
> **Run:** Execute the code in Colab.  
> **Observe:** Check for exceptions, warnings that affect correctness, invalid K values, missing-data failures, and whether the printed result is understandable.  
> **Fix:** If a problem appears, revise only what is necessary, rerun the complete script, and show the successful final output. Include a concise note describing the change.

### Iteration note
The initial approach used a fixed `K=3`. The revised version evaluates `K=2,3,4`, uses silhouette score to select among them, validates required columns, fills missing numeric values with medians, and sets `n_init=10` explicitly.

## ReACT stages shown in this notebook

**Plan summary:** Validate the input first, preprocess the numeric features, compare several cluster counts, select the strongest tested K using silhouette score, fit the final model, and summarize the segments.

**Act:** Run the Python cell below.

**Observe:** Confirm that all silhouette scores print, a best K is selected, cluster assignments are produced, and a CSV file is saved.

**Fix:** The final code includes the fixes described above so the notebook runs without depending on a manually chosen K.

In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Sample customer data
customer_data = {
    'annual_spending': [500, 1200, 300, 1500, 800, 200, 1000, 600, 1300, 400],
    'purchase_frequency': [5, 12, 3, 15, 8, 2, 10, 6, 13, 4],
    'age': [25, 34, 45, 28, 52, 36, 41, 29, 47, 33],
    'region': ['North', 'South', 'West', 'East', 'South', 'North', 'West', 'East', 'South', 'North']
}

df = pd.DataFrame(customer_data)
required_features = ['annual_spending', 'purchase_frequency', 'age']

try:
    missing_columns = [col for col in required_features if col not in df.columns]
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")
    if len(df) < 4:
        raise ValueError("At least 4 rows are required to test multiple cluster counts.")

    X = df[required_features].apply(pd.to_numeric, errors='coerce')
    X = X.fillna(X.median(numeric_only=True))

    if X.isna().any().any():
        raise ValueError("One or more required numeric columns could not be cleaned.")

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    candidate_ks = [k for k in range(2, 5) if k < len(df)]
    scores = {}

    for k in candidate_ks:
        model = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = model.fit_predict(X_scaled)
        scores[k] = silhouette_score(X_scaled, labels)

    best_k = max(scores, key=scores.get)
    final_model = KMeans(n_clusters=best_k, random_state=42, n_init=10)
    df['cluster'] = final_model.fit_predict(X_scaled)

    cluster_summary = df.groupby('cluster')[required_features].mean().round(2)

    print("Silhouette scores by K:")
    for k, score in scores.items():
        print(f"  K={k}: {score:.3f}")

    print(f"\nSelected K: {best_k}")
    print("\nCluster summary:")
    print(cluster_summary)

    output_file = 'customer_segments_react.csv'
    df.to_csv(output_file, index=False)
    print(f"\nSaved final customer segments to: {output_file}")

except Exception as e:
    print("The script could not complete:", e)


Silhouette scores by K:
  K=2: 0.453
  K=3: 0.420
  K=4: 0.359

Selected K: 2

Cluster summary:
         annual_spending  purchase_frequency   age
cluster                                           
0                  400.0                 4.0  33.6
1                 1160.0                11.6  40.4

Saved final customer segments to: customer_segments_react.csv


## Observation and fix note
The successful run prints a silhouette score for every tested K, selects the best tested K, prints the cluster averages, and writes the segmented customers to CSV. Compared with the initial fixed-K approach, the final version makes the cluster choice data-driven and includes basic input/error handling.